# Passo 1: Preparação Simplificada dos Dados (Padding)

In [1]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Amostra de comentários do e-commerce
comentarios = [
    "O produto é maravilhoso, super recomendo",
    "Entrega rápida e excelente atendimento",
    "Detestei o produto, veio quebrado e estragado",
    "Péssima qualidade, não comprem de jeito nenhum"
]

# 1 = Positivo, 0 = Negativo
labels = np.array([1, 1, 0, 0])

# Criando o Tokenizer e convertendo os textos em números
tokenizer = Tokenizer(num_words=1000, oov_token="<UNK>")
tokenizer.fit_on_texts(comentarios)
sequencias = tokenizer.texts_to_sequences(comentarios)

# Padronizando o tamanho das frases para ter exatamente 6 palavras (Padding)
dados_prontos = pad_sequences(sequencias, maxlen=6, padding='post')

print("Frases transformadas em sequências numéricas fixas:\n", dados_prontos)

Frases transformadas em sequências numéricas fixas:
 [[ 2  3  5  6  7  8]
 [ 9 10  4 11 12  0]
 [ 2  3 14 15  4 16]
 [18 19 20 21 22 23]]


# Passo 2: Construindo a Rede Neural Sequencial (Opção A: RNN Clássica)

In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# [Sequência de IDs Inteiros] -> (1. Embedding) -> (2. SimpleRNN) -> (3. Dense) -> [Sentimento (0 ou 1)]

vocab_size = 1000  # Tamanho máximo do vocabulário
max_len = 6        # Tamanho de cada sequência de entrada

model_rnn = Sequential([
    # Camada 1: Transforma IDs em vetores contínuos (Word Embeddings)
    Embedding(input_dim=vocab_size, output_dim=16, input_length=max_len),
    
    # Camada 2: A Célula Recorrente Simples (RNN) que processa palavra por palavra
    SimpleRNN(units=8),
    
    # Camada 3: Neurônio de saída (Sigmóide mapeia o resultado entre 0 e 1)
    Dense(units=1, activation='sigmoid')
])

model_rnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_rnn.summary()

C:\Users\paulo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# Passo 2: Construindo a Rede Neural Sequencial (Opção B: LSTM)

In [3]:
from tensorflow.keras.layers import LSTM

model_lstm = Sequential([
    Embedding(input_dim=vocab_size, output_dim=16, input_length=max_len),
    
    # Mudança conceitual: Agora o fluxo possui portões de retenção de longo prazo
    LSTM(units=8),
    
    Dense(units=1, activation='sigmoid')
])

model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_lstm.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# Passo 3: Treinamento e Predição

In [4]:
# Treinando o modelo LSTM com os nossos dados de exemplo
model_lstm.fit(dados_prontos, labels, epochs=5, verbose=0)
model_rnn.fit(dados_prontos, labels, epochs=5, verbose=0)

# Testando uma frase nova
nova_frase = ["O atendimento foi excelente"]
nova_seq = tokenizer.texts_to_sequences(nova_frase)
nova_seq_padded = pad_sequences(nova_seq, maxlen=6, padding='post')

predicao = model_rnn.predict(nova_seq_padded)
print(f"\nProbabilidade de ser um comentário positivo: {predicao[0][0]:.4f}")

predicao = model_lstm.predict(nova_seq_padded)
print(f"\nProbabilidade de ser um comentário positivo: {predicao[0][0]:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step



Probabilidade de ser um comentário positivo: 0.4790
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step



Probabilidade de ser um comentário positivo: 0.5040


# Transformers - Passo 1: Instanciação Simples de um Pipeline

In [5]:
# Instale a biblioteca necessária
# !pip install transformers

from transformers import pipeline

# Criando um classificador de análise de sentimento usando um Transformer padrão (BERT)
classificador = pipeline("sentiment-analysis")

# Testando uma frase que confunde os modelos clássicos (como negações ou sarcasmo)
resultado = classificador("Eu não acho que o produto seja ruim, pelo contrário, superou minhas expectativas.")
print(resultado)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

C:\Users\paulo\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\paulo\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'NEGATIVE', 'score': 0.9859282970428467}]


# Transformers - Passo 2: Abrindo a Caixa-Preta

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

nome_modelo = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(nome_modelo)
modelo = AutoModelForSequenceClassification.from_pretrained(nome_modelo)

frase = "Transformers are amazing at context."

# 1. Veja como o Tokenizer transforma o texto em IDs numéricos e adiciona tokens especiais
inputs = tokenizer(frase, return_tensors="pt")
print("Tokens convertidos em IDs numéricos:\n", inputs["input_ids"])

# 2. Passe os IDs pelo modelo para obter as predições (Logits)
with torch.no_grad():
    outputs = modelo(**inputs)

print("\nSaída bruta matemática do modelo (Logits):\n", outputs.logits)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

C:\Users\paulo\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\paulo\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Tokens convertidos em IDs numéricos:
 tensor([[  101, 19081,  2024,  6429,  2012,  6123,  1012,   102]])

Saída bruta matemática do modelo (Logits):
 tensor([[-4.2318,  4.5300]])
